In [ ]:
import sys
import os
from dotenv import load_dotenv
load_dotenv()

sys.path.append("..")

from pytorch_lightning.loggers import CometLogger
from pathlib import Path
from datetime import datetime, timedelta
from lib.wpt_utils import load_wavelets
from training.logging_utils import log_metrics
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pytorch_lightning as pl

from training.training_pipeline_v2 import run_pipeline
from training.objects import CosinePipelineOutput

In [ ]:
COMET_API_KEY = os.environ.get("COMET_API_KEY")

In [ ]:
SEQ_1_PATH = Path("../data/wpts/sequence_1_db35")
SEQ_2_PATH = Path("../data/wpts/sequence_2_db35")
SEQ_3_PATH = Path("../data/wpts/sequence_3_db35")

In [ ]:
data_seq_1 = load_wavelets(SEQ_1_PATH)
data_seq_2 = load_wavelets(SEQ_2_PATH)
data_test = load_wavelets(SEQ_3_PATH)

In [ ]:
data_train, data_valid = data_seq_2, data_seq_1

# Single

In [ ]:
params = {
    "normalize": True,
    "batch_size": 128,
    "num_epochs": 100,
    "lr": 1e-3,
    "hidden_dim": 16,
    "num_layers": 3,
    "activation": "relu",
    "fs": 256,
    "bandwidth": 113,
    "beta": 0.001,
    "f_min": 7,
    "latent_dim": 6,
    "max_level": 6,
    "min_level": 3,
}

In [ ]:
logger = CometLogger(
    api_key=COMET_API_KEY,
project_name=os.environ.get("COMET_PROJECT_COSINE"),
workspace=os.environ.get("COMET_WORKSPACE"),
)

In [ ]:
logger.log_hyperparams(params)

In [ ]:
train_output, valid_output, test_output = run_pipeline(
    data_train=data_train,
    data_valid=data_valid,
    params=params,
    logger=logger,
    return_model=True,
    data_test=data_test,
)

In [ ]:
def extract_metadata_df(output):
    x_list = []
    for m_list in output.metadata:
        x = pd.DataFrame([m.model_dump() for m in m_list])
        x_list.append(x)
    return pd.concat(x_list)


In [ ]:
output = train_output
df = extract_metadata_df(output)
df["cable"] = df["cable"].astype(int)

In [ ]:
# import scienceplots

# plt.style.use(["science", "ieee", "grid"])

In [ ]:
plt.style.use("default")

In [ ]:
for cable in [0, 1, 2, 3]:
    tensions = df.loc[df["cable"] == cable, "tension"].to_list()
    dates = df.loc[df["cable"] == cable, "sampled_at"].to_list()
    z = output.embeddings[:, cable, :]
    x = output.inputs[:, cable, :]
    x_rec = output.reconstructions[:, cable, :]
    index = np.arange(x.shape[1])

    mse = np.sqrt(np.mean((x - x_rec) ** 2))

    # # Inputs
    # fig = plt.figure(figsize=(2, 2))
    # ax = fig.add_subplot(1, 1, 1)    
    # mesh = ax.pcolormesh(index, tensions, x, cmap="Blues", dpi=80)
    # cbar = plt.colorbar(mesh, ax=ax)
    # cbar.set_label(r"$\mathbf{x}_{" + str(cable+1) + r"}$")
    # ax.set_xlabel("Sub-band index")
    # ax.set_ylabel("Tension (N)")
    # logger.experiment.log_figure(
    #     figure=fig, figure_name=f"input_cable_{cable+1}"
    # )
    # plt.close("all")

    # # Reconstructions
    # fig = plt.figure(figsize=(2, 2))
    # ax = fig.add_subplot(1, 1, 1)
    # mesh = ax.pcolormesh(index, tensions, x_rec, cmap="Blues", dpi=80)
    # cbar = plt.colorbar(mesh, ax=ax)
    # cbar.set_label(r"$\mathbf{y}_{" + str(cable+1) + r"}$")
    # ax.set_xlabel("Sub-band index")
    # ax.set_ylabel("Tension (N)")
    # logger.experiment.log_figure(
    #     figure=fig, figure_name=f"reconstruction_cable_{cable+1}"
    # )
    # plt.close("all")

In [ ]:
logger.experiment.end()

In [ ]:
df_train = pd.DataFrame([o.model_dump() for o in output.train_vae_results])
df_valid = pd.DataFrame([o.model_dump() for o in output.valid_vae_results])
df_test = pd.DataFrame([o.model_dump() for o in output.test_vae_results])

In [ ]:
def extract_arrays(df: pd.DataFrame) -> tuple[np.ndarray, np.ndarray, list[datetime]]:
    x = []
    t = []
    for cable in [0, 1, 2, 3]:
        df_cable = df.loc[df.cable == cable].copy()
        df_cable.sort_values("sampled_at")
        x.append(np.array(df_cable["values"].to_list()))
        sampled_at = df_cable["sampled_at"].to_list()
        t.append(np.array(df_cable["tension"].to_list()))
    x = np.array(x)
    t = np.array(t)
    return x, t, sampled_at

In [ ]:
x_train, t_train, s_train = extract_arrays(df_train)
x_valid, t_valid, s_valid = extract_arrays(df_valid)
x_test, t_test, s_test = extract_arrays(df_test)

In [ ]:
# n_latent = x_train.shape[2]
# n_cables = x_train.shape[0]

In [ ]:
dist_train = pd.DataFrame([o.model_dump() for o in output.train_dist_results])
dist_valid = pd.DataFrame([o.model_dump() for o in output.valid_dist_results])
dist_test = pd.DataFrame([o.model_dump() for o in output.test_dist_results])

In [ ]:
d_train, t_train, _ = extract_arrays(dist_train)
d_valid, t_valid, _ = extract_arrays(dist_valid)
d_test, t_test, _ = extract_arrays(dist_test)

d_train = d_train.squeeze()
d_valid = d_valid.squeeze()
d_test = d_test.squeeze()

In [ ]:
plt.figure(figsize=(8, 8))
for i in range(4):
    plt.subplot(2, 2, i + 1)
    plt.plot(t_train[i, :], d_train[i, :], label=f"Train Cable {i}")
    plt.plot(t_valid[i, :], d_valid[i, :], label=f"Valid Cable {i}")
    plt.plot(t_test[i, :], d_test[i, :], label=f"Test Cable {i}")

In [ ]:
plt.figure(figsize=(12, 8))
for i in range(4):
    # plt.subplot(2,2,i+1)
    plt.plot(t_train[i, :], d_train[i, :], label=f"Train Cable {i}")
    plt.plot(t_valid[i, :], d_valid[i, :], label=f"Valid Cable {i}")
    plt.plot(t_test[i, :], d_test[i, :], label=f"Test Cable {i}")

In [ ]:
plt.figure(figsize=(12, 12))
for j in range(4):
    plt.subplot(2, 2, j + 1)
    for i in range(4):
        plt.plot(t_train[i, :], x_train[i, :, j], label=f"Train Cable {i}")
        plt.plot(t_valid[i, :], x_valid[i, :, j], label=f"Valid Cable {i}")
        plt.plot(t_test[i, :], x_test[i, :, j], label=f"Test Cable {i}")

# Save data

In [ ]:
import json
from pydantic import BaseModel

In [ ]:
path_train = Path("../data/embeddings/sequence_train")
path_valid = Path("../data/embeddings/sequence_valid")
path_test = Path("../data/embeddings/sequence_test")

In [ ]:
x_train.shape, t_train.shape, len(s_train)

In [ ]:
class Embedding(BaseModel):
    cable: int
    tension: float
    values: list[float]

In [ ]:
class SampleEmbeddings(BaseModel):
    sampled_at: datetime
    c0: Embedding
    c1: Embedding
    c2: Embedding
    c3: Embedding

In [ ]:
for k in range(len(s_train)):
    sample = SampleEmbeddings(
        sampled_at=s_train[k],
        c0=Embedding(cable=0, tension=float(t_train[0, k]), values=x_train[0, k, :].tolist()),
        c1=Embedding(cable=1, tension=float(t_train[1, k]), values=x_train[1, k, :].tolist()),
        c2=Embedding(cable=2, tension=float(t_train[2, k]), values=x_train[2, k, :].tolist()),
        c3=Embedding(cable=3, tension=float(t_train[3, k]), values=x_train[3, k, :].tolist()),
    )
    filename = s_train[k].isoformat().replace(":", "-")
    data = sample.model_dump()
    data["sampled_at"] = data["sampled_at"].isoformat()
    with open(path_train / f"{filename}.json", "w") as f:
        json.dump(data, f, indent=4)

In [ ]:
for k in range(len(s_valid)):
    sample = SampleEmbeddings(
        sampled_at=s_valid[k],
        c0=Embedding(cable=0, tension=float(t_valid[0, k]), values=x_valid[0, k, :].tolist()),
        c1=Embedding(cable=1, tension=float(t_valid[1, k]), values=x_valid[1, k, :].tolist()),
        c2=Embedding(cable=2, tension=float(t_valid[2, k]), values=x_valid[2, k, :].tolist()),
        c3=Embedding(cable=3, tension=float(t_valid[3, k]), values=x_valid[3, k, :].tolist()),
    )
    filename = s_valid[k].isoformat().replace(":", "-")
    data = sample.model_dump()
    data["sampled_at"] = data["sampled_at"].isoformat()
    with open(path_valid / f"{filename}.json", "w") as f:
        json.dump(data, f, indent=4)

In [ ]:
for k in range(len(s_test)):
    sample = SampleEmbeddings(
        sampled_at=s_test[k],
        c0=Embedding(cable=0, tension=float(t_test[0, k]), values=x_test[0, k, :].tolist()),
        c1=Embedding(cable=1, tension=float(t_test[1, k]), values=x_test[1, k, :].tolist()),
        c2=Embedding(cable=2, tension=float(t_test[2, k]), values=x_test[2, k, :].tolist()),
        c3=Embedding(cable=3, tension=float(t_test[3, k]), values=x_test[3, k, :].tolist()),
    )
    filename = s_test[k].isoformat().replace(":", "-")
    data = sample.model_dump()
    data["sampled_at"] = data["sampled_at"].isoformat()
    with open(path_test / f"{filename}.json", "w") as f:
        json.dump(data, f, indent=4)